# Train ICARUS VAE — GPU slice A

**Kernel:** `conda env:.conda-diffusion` (one GPU / MIG slice)

Companion: `04b_TrainCAE_ICARUS.ipynb` on the other slice.

Data: `/exp/sbnd/data/users/gputnam/DNN-ROI-images/`  
Flags: `configs/train_flags/vae_icarus.sh`  
Logs: scratch `.../training/vae/icarus/` (synced to `DATA_ROOT/training/vae/icarus/`)

**Before launch:** stop leftover 01a/01b trainers and clear the stop flag (on EAF):
```bash
bash train/stop_running_trainers.sh
rm -f /exp/sbnd/app/users/munjung/anomaly-detection/STOP_TRAINING
```

### Modes (set `MODE` in the next cell)
| `MODE` | What happens |
|--------|----------------|
| `"dry"` | Write `launch.sh` only — **no training** (default) |
| `"smoke"` | Short GPU run (~5 steps) → tiny checkpoint |
| `"full"` | Train to `MAX_STEPS` (263000) |


In [15]:
from __future__ import annotations

import importlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
if "configs.paths" in sys.modules:
    importlib.reload(sys.modules["configs.paths"])

from configs.paths import (
    DATA_ROOT,
    GPUTNAM_DNN_ROI,
    describe_environment,
    ensure_layout,
    existing_scratch_pools,
    resolve_ae_training_run_root,
)

ensure_layout()
env = describe_environment()
print(json.dumps(env, indent=2))
print("scratch pools:", [str(p) for p in existing_scratch_pools()] or ["<none>"])
print("DNN-ROI:", GPUTNAM_DNN_ROI, "exists=", GPUTNAM_DNN_ROI.is_dir())
assert env.get("on_eaf") or existing_scratch_pools(), (
    "Expected EAF scratch. Open this notebook on jupyter-munjung with the diffusion kernel."
)

DIFFUSION_ROOT = APP_ROOT / "train" / "diffusion-anomaly"
FLAG_SCRIPT = APP_ROOT / "configs" / "train_flags" / "vae_icarus.sh"
AE_TYPE = "vae"
assert FLAG_SCRIPT.is_file(), FLAG_SCRIPT
assert "icarus" in FLAG_SCRIPT.name and "sbnd" not in FLAG_SCRIPT.name, FLAG_SCRIPT

# --- ICARUS samples (NOT SBND) ---
# Flag DATADIR → /exp/sbnd/data/users/gputnam/DNN-ROI-images/
# Format: event_*/deconvolved_signal (ICARUS). SBND raw would be event_*/raw.
DATA_DIR = GPUTNAM_DNN_ROI
assert DATA_DIR.is_dir(), DATA_DIR
h5s = sorted(DATA_DIR.glob("*.h5"))
assert h5s, f"No .h5 under {DATA_DIR}"
import h5py
with h5py.File(h5s[0], "r") as _f:
    _ev = next(iter(_f.keys()))
    assert "deconvolved_signal" in _f[_ev], (
        f"{h5s[0]} looks like SBND/raw, not ICARUS DNN-ROI (missing deconvolved_signal)"
    )
print(f"TRAINING DATA (ICARUS DNN-ROI): {DATA_DIR}")
print(f"  n_h5={len(h5s)}  example={h5s[0].name}  format=deconvolved_signal")

# --- controls ---
# MODE: "dry" | "smoke" | "full"
MODE = "full"          # ← launching full ICARUS VAE training
RESUME = True
MAX_STEPS = "263000"
BATCH_SIZE = None      # None → flag script (16); set 4/8 on tight MIG

DRY_RUN = MODE == "dry"
SMOKE_TEST = MODE == "smoke"
print(f"MODE={MODE}  DRY_RUN={DRY_RUN}  SMOKE_TEST={SMOKE_TEST}")


{
  "hostname": "jupyter-munjung",
  "on_eaf": true,
  "app_root": "/exp/sbnd/app/users/munjung/anomaly-detection",
  "data_root": "/exp/sbnd/data/users/munjung/anomaly-detection",
  "data_root_exists": true,
  "scratch_pools": [
    "/scratch/7DayLifetime"
  ],
  "scratch_root": "/scratch/7DayLifetime/munjung/anomaly-detection",
  "scratch_exists": true,
  "scratch_icarus": "/scratch/7DayLifetime/munjung/ICARUS",
  "scratch_icarus_exists": true,
  "diffusion_code_root": "/exp/sbnd/app/users/munjung/anomaly-detection/train/diffusion-anomaly",
  "default_checkpoint": "/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt",
  "default_checkpoint_exists": true,
  "vae_sbnd_checkpoint": "None",
  "cae_sbnd_checkpoint": "None",
  "vae_icarus_checkpoint": "None",
  "cae_icarus_checkpoint": "None",
  "gputnam_dnn_roi": "/exp/sbnd/data/users/gputnam/DNN-ROI-images",
  "gputnam_dnn_roi_exists": true
}
scratch pools: ['/scratch/7DayLifetime']
DNN-ROI: /exp/sbnd/data/user

In [16]:
stop = APP_ROOT / "STOP_TRAINING"
if stop.is_file():
    stop.unlink()
    print(f"removed stop file {stop}")

LOG_DIR = resolve_ae_training_run_root(AE_TYPE, "icarus")
DURABLE = DATA_ROOT / "training" / AE_TYPE / "icarus"
DURABLE.mkdir(parents=True, exist_ok=True)
print("LOG_DIR ", LOG_DIR)
print("DURABLE ", DURABLE)
print("DATA_DIR (ICARUS)", DATA_DIR)


def latest_resume(log_dir: Path, durable: Path) -> Path | None:
    cands = []
    for d in (log_dir, durable):
        if not d.is_dir():
            continue
        cands += sorted(d.glob("brats2update*.pt"))
        cands += sorted(d.glob("model*.pt"))
    return cands[-1] if cands else None


resume_ckpt = latest_resume(LOG_DIR, DURABLE) if RESUME else None
print("resume:", resume_ckpt)

# Explicit ICARUS data paths last so they win over anything in the flag script.
extras = [
    f"--data_dir {DATA_DIR}",
    f"--validation_dir {DATA_DIR}",
    f"--max_steps {MAX_STEPS}",
]
if BATCH_SIZE is not None:
    mb = min(int(BATCH_SIZE), 4)
    extras += [f"--batch_size {BATCH_SIZE}", f"--microbatch {mb}"]
if SMOKE_TEST:
    extras = [
        f"--data_dir {DATA_DIR}",
        f"--validation_dir {DATA_DIR}",
        "--max_steps 5",
        "--save_interval 5",
        "--validation_interval 5",
        "--plot_interval 1000000",
        "--batch_size 2",
        "--microbatch 2",
        "--log_interval 1",
    ]

resume_flag = f"--resume_checkpoint {resume_ckpt}" if resume_ckpt else ""
cmd = f"""set -euo pipefail
source {FLAG_SCRIPT}
cd {DIFFUSION_ROOT}
python3 scripts/autoencoder_train.py $AE_TRAIN_FLAGS \\
  --log_dir {LOG_DIR} \\
  {resume_flag} {' '.join(extras)}
"""
print(cmd)
assert "DNN-ROI-images" in cmd, "launch cmd must use ICARUS DNN-ROI data"
assert "training-SBND" not in cmd and "filelists" not in cmd, "SBND paths leaked into launch cmd"
WRAPPER = LOG_DIR / "launch.sh"
WRAPPER.write_text(cmd)
WRAPPER.chmod(0o755)
print("wrote", WRAPPER)


LOG_DIR  /scratch/7DayLifetime/munjung/anomaly-detection/training/vae/icarus
DURABLE  /exp/sbnd/data/users/munjung/anomaly-detection/training/vae/icarus
DATA_DIR (ICARUS) /exp/sbnd/data/users/gputnam/DNN-ROI-images
resume: None
set -euo pipefail
source /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/vae_icarus.sh
cd /exp/sbnd/app/users/munjung/anomaly-detection/train/diffusion-anomaly
python3 scripts/autoencoder_train.py $AE_TRAIN_FLAGS \
  --log_dir /scratch/7DayLifetime/munjung/anomaly-detection/training/vae/icarus \
   --data_dir /exp/sbnd/data/users/gputnam/DNN-ROI-images --validation_dir /exp/sbnd/data/users/gputnam/DNN-ROI-images --max_steps 263000

wrote /scratch/7DayLifetime/munjung/anomaly-detection/training/vae/icarus/launch.sh


In [ ]:
log_file = LOG_DIR / "train.log"
run_env = os.environ.copy()
if SMOKE_TEST:
    run_env["DIFFUSION_TRAINING_TEST"] = "1"

print("MODE", MODE, "DRY_RUN", DRY_RUN, "SMOKE_TEST", SMOKE_TEST, "MAX_STEPS", MAX_STEPS)
if DRY_RUN:
    print("(dry) not launching — set MODE='full' or MODE='smoke' and re-run from controls.")
else:
    stop = APP_ROOT / "STOP_TRAINING"
    if stop.is_file():
        stop.unlink()
        print("removed", stop)
    print("launching", WRAPPER)
    print("This cell blocks until training finishes (or you Interrupt the kernel).")
    with open(log_file, "a") as lf:
        lf.write(f"\n# launch {time.asctime()} AE={AE_TYPE} mode={MODE}\n")
    ret = subprocess.run(["bash", str(WRAPPER)], cwd=str(DIFFUSION_ROOT), env=run_env)
    print("exit", ret.returncode)
    for pattern in (
        "ema_*.pt", "emabrats2update_*.pt", "brats2update*.pt",
        "model*.pt", "progress.csv", "log.txt",
    ):
        for src in LOG_DIR.glob(pattern):
            dest = DURABLE / src.name
            if (not dest.exists()) or src.stat().st_mtime > dest.stat().st_mtime:
                dest.write_bytes(src.read_bytes())
                print("synced", dest)


MODE full DRY_RUN False SMOKE_TEST False MAX_STEPS 263000
launching /scratch/7DayLifetime/munjung/anomaly-detection/training/vae/icarus/launch.sh
This cell blocks until training finishes (or you Interrupt the kernel).
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
port2 50747
setup_dist: backend=gloo CUDA_VISIBLE_DEVICES=0 cuda_available=True device_count=1
setup_dist: device0= NVIDIA A100 80GB PCIe MIG 2g.20gb
Logging to /scratch/7DayLifetime/munjung/anomaly-detection/training/vae/icarus
creating autoencoder (vae)...
creating data loader...
training...


0it [00:00, ?it/s]

-------------------------
| grad_norm  | 2.42     |
| kld        | 0.000238 |
| kld_q0     | 0.000238 |
| loss       | 0.0897   |
| loss_q0    | 0.0897   |
| mse        | 0.0897   |
| mse_q0     | 0.0897   |
| param_norm | 89.3     |
| samples    | 16       |
| step       | 0        |
-------------------------


50it [02:07,  2.28s/it]

--------------------------
| grad_norm   | 0.405    |
| kld         | 0.00633  |
| kld_q0      | 0.00594  |
| loss        | 0.0226   |
| loss_q0     | 0.0209   |
| mse         | 0.0226   |
| mse_q0      | 0.0209   |
| param_norm  | 89.4     |
| samples     | 816      |
| step        | 50       |
| val-kld_q0  | 0.00825  |
| val-loss_q0 | 0.0313   |
| val-mse_q0  | 0.0313   |
--------------------------


...

8150it [5:47:39,  2.24s/it]

--------------------------
| grad_norm   | 0.0631   |
| kld         | 0.0254   |
| kld_q0      | 0.0256   |
| loss        | 0.00961  |
| loss_q0     | 0.01     |
| mse         | 0.00961  |
| mse_q0      | 0.01     |
| param_norm  | 98.2     |
| samples     | 1.3e+05  |
| step        | 8.15e+03 |
| val-kld_q0  | 0.0247   |
| val-loss_q0 | 0.00761  |
| val-mse_q0  | 0.00761  |
--------------------------


8200it [5:49:47,  2.21s/it]

--------------------------
| grad_norm   | 0.0607   |
| kld         | 0.023    |
| kld_q0      | 0.0235   |
| loss        | 0.015    |
| loss_q0     | 0.013    |
| mse         | 0.015    |
| mse_q0      | 0.013    |
| param_norm  | 98.3     |
| samples     | 1.31e+05 |
| step        | 8.2e+03  |
| val-kld_q0  | 0.0207   |
| val-loss_q0 | 0.0252   |
| val-mse_q0  | 0.0252   |
--------------------------


8215it [5:50:26,  2.34s/it]

## Monitor

```bash
tail -f /scratch/7DayLifetime/munjung/anomaly-detection/training/vae/icarus/train.log
```

After a good run, prefer EMA under `DATA_ROOT/training/vae/icarus/` for
`inference/07_BaselineVAE_CAE_ICARUS.ipynb`.
